> **Trust but verify.** Even examples Rick shares and AI-generated suggestions can be wrong or unsafe. Read code, commands and configuration before running them; use disposable copies, keep backups, and verify results. Don't let an example eat your homework.

# A sentence through PyTorch

Goal zero: input → local model execution on CPU → output. This is an English sentiment example, not a language-generating assistant or a training lesson. You can watch the demonstration without installing anything or creating an account.

First follow [setup.md](setup.md). The load cell downloads and caches the tokenizer and pretrained weights from [M-FAC/bert-tiny-finetuned-sst2](https://huggingface.co/M-FAC/bert-tiny-finetuned-sst2/tree/41ad6709ec46b414749b37daf49cf5ca1c7dba7c), pinned to that revision. Subsequent computation happens locally. No text is sent to an inference service.


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_ID = "M-FAC/bert-tiny-finetuned-sst2"
REVISION = "41ad6709ec46b414749b37daf49cf5ca1c7dba7c"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=REVISION, token=False, trust_remote_code=False
)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID, revision=REVISION, token=False, trust_remote_code=False, use_safetensors=False, weights_only=True
).to("cpu")
model.eval()
print("Model ready on", next(model.parameters()).device)


In [ ]:
text = "This movie was wonderful."
inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)

with torch.inference_mode():
    logits = model(**inputs).logits
    scores = torch.softmax(logits, dim=-1)[0]

# SST-2 class order: 0 = negative, 1 = positive.
labels = ("negative", "positive")
prediction = labels[int(scores.argmax())]
print("Input:", text)
print("Prediction:", prediction)
print("Scores:", {label: round(float(score), 4) for label, score in zip(labels, scores)})


### Follow these calls into source

When you run a cell, JupyterLab sends an [execution request](https://github.com/jupyterlab/jupyterlab/blob/v4.4.9/packages/services/src/kernel/default.ts#L786-L813). The server relays it to [ipykernel's execution handler](https://github.com/ipython/ipykernel/blob/v6.30.1/ipykernel/kernelbase.py#L721-L796); [the Python kernel](https://github.com/ipython/ipykernel/blob/v6.30.1/ipykernel/ipkernel.py#L370-L478) asks IPython to run the code. The browser does not execute this Python model.

In our prediction cell, [`torch.inference_mode`](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/autograd/grad_mode.py#L212-L280) controls autograd bookkeeping. Calling `model(**inputs)` goes through [PyTorch's module dispatch](https://github.com/pytorch/pytorch/blob/v2.8.0/torch/nn/modules/module.py#L1769-L1790) to [Transformers' `BertForSequenceClassification.forward`](https://github.com/huggingface/transformers/blob/v4.56.2/src/transformers/models/bert/modeling_bert.py#L1460-L1507). [`torch.softmax`](https://github.com/pytorch/pytorch/blob/v2.8.0/aten/src/ATen/native/SoftMax.cpp#L411-L423) turns logits into the displayed scores; this native entry delegates to the device implementation. Model architecture code and downloaded trained weights have different sources.

These are selected implementation entry points, not every internal call frame. Links use the tested release tags. The [component mapping notebook](component-mapping.ipynb) connects them to process boundaries, repositories and upstream projects.


Try changing the sentence and rerunning the last cell. The result is the model's prediction, not proof of what a person feels; these scores are not calibrated confidence estimates. Inputs longer than 128 tokens are truncated.

For a fresh run, use **Kernel → Restart Kernel and Run All Cells**. Stop here for goal zero; instrumentation and the component walkthrough will follow after review.
